# 03 Custom Algorithms

Load a small custom `score_pair` module, score archive pairs, evaluate a dataset, and inspect reproducibility metadata attached to the results.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

MATHEEL_REF = "v0.5.1"


def run(*args, cwd=None):
    subprocess.check_call(list(args), cwd=cwd)

run(sys.executable, "-m", "pip", "install", "jedi>=0.16,<1")
run("rm", "-rf", "/content/matheel")
run("git", "clone", "https://github.com/FahadEbrahim/matheel.git", "/content/matheel")
run("git", "checkout", MATHEEL_REF, cwd="/content/matheel")
run(sys.executable, "-m", "pip", "install", "-e", ".[all]", cwd="/content/matheel")
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matheel_mplconfig")
print(f"Matheel installed from {MATHEEL_REF}")


In [ ]:
%cd /content/matheel
from pathlib import Path
from tempfile import TemporaryDirectory

import pandas as pd

from matheel.algorithms import score_pair_with_algorithm, score_source_pairs_with_algorithm
from matheel.datasets import write_pair_dataset
from matheel.evaluation import evaluate_pair_dataset

In [ ]:
def write_custom_algorithm(path):
    Path(path).write_text(
        "
".join([
            "def prepare_dataset(dataset, bias=0.0, **kwargs):",
            "    _ = (dataset, kwargs)",
            "    return {'bias': float(bias)}",
            "",
            "def score_pair(code_a, code_b, dataset_context=None, row=None, **kwargs):",
            "    _ = (row, kwargs)",
            "    base = 1.0 if code_a.strip() == code_b.strip() else 0.0",
            "    return base + dataset_context['bias']",
        ]),
        encoding="utf-8",
    )

In [ ]:
workspace = Path("/content/matheel_custom_algorithm_demo")
workspace.mkdir(parents=True, exist_ok=True)
algorithm_path = workspace / "my_algorithm.py"
write_custom_algorithm(algorithm_path)

source_root = workspace / "codes"
source_root.mkdir(exist_ok=True)
(source_root / "a.py").write_text("print(1)
", encoding="utf-8")
(source_root / "b.py").write_text("print(1)
", encoding="utf-8")
(source_root / "c.py").write_text("print(2)
", encoding="utf-8")

archive_scores = score_source_pairs_with_algorithm(
    source_root,
    algorithm=algorithm_path,
    algorithm_options={"bias": 0.1},
    number_results=3,
)
display(archive_scores[["file_name_1", "file_name_2", "similarity_score"]])
print(archive_scores.attrs["algorithm"])

In [ ]:
dataset_root = workspace / "pairs"
dataset = write_pair_dataset(
    dataset_root,
    files=pd.DataFrame([
        {"file_id": "a", "text": "print(1)", "suffix": ".py"},
        {"file_id": "b", "text": "print(1)", "suffix": ".py"},
        {"file_id": "c", "text": "print(2)", "suffix": ".py"},
    ]),
    pairs=pd.DataFrame([
        {"left_id": "a", "right_id": "b", "label": 1},
        {"left_id": "a", "right_id": "c", "label": 0},
    ]),
)
scored, metrics = evaluate_pair_dataset(
    dataset,
    threshold=0.5,
    algorithm=algorithm_path,
    algorithm_options={"bias": 0.1},
)
display(scored)
print(metrics)

In [ ]:
direct_score = score_pair_with_algorithm(
    "print(1)",
    "print(1)",
    algorithm_path,
    algorithm_options={"bias": 0.1},
    dataset_context={"bias": 0.1},
)
print(round(direct_score, 4))